### QuSciTech-Labs — Navigation

[Public Labs](https://github.com/jopaneur/quscitech-labs) ·
[Full Edition Access](https://github.com/jopaneur/quscitech-labs#-full-edition-kdp) ·
[Private Repo](https://github.com/jopaneur/quscitech-labs-full) ·
[QuSciTech.com](https://www.quscitech.com) ·
[The Quantum AI Book (QAIS)](https://www.amazon.com/dp/placeholder) ·

DOI: [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.17212825.svg)](https://doi.org/10.5281/zenodo.17212825)

### E.2 Lab 3 — BB84 Quantum Key Distribution — Error Rate Comparison

### Lab Access and Execution Guide
This guide explains how to run and explore the hands-on quantum computing labs that accompany the book  
**Quantum AI Systems: Theory, Architecture, and Applications** (Professional Volume).  

The labs are an integral part of the MyQuantumBook project, designed to reinforce key concepts from the chapters through interactive exploration. They are built for execution on **Google Colab** and **IBM Quantum backends** using **Qiskit**, and follow the IEEE-compliant figure, caption, and documentation standards described in the text.  

Each lab is cross-referenced to its corresponding chapter and appendix figure (Appendix E), ensuring reproducibility and scholarly traceability.  

**Getting Started**
1. Launch the notebook in Google Colab using the provided badge.
2. Run the setup cells to install Qiskit:
   `!pip install qiskit`

**Using IBM Quantum Systems**
1. Sign up at https://quantum.ibm.com and create an API token.
2. Run the IBMQ setup cell.
3. Replace 'MY_API_TOKEN' with your real token (only needed once).
4. Select backends using `provider.get_backend('ibmq_qasm_simulator')` or others.

**Lab Structure**
Each code section aligns with a chapter from the book.
- Modify and re-run code blocks.
- View circuits with `.draw()`.
- Apply to custom inputs to deepen your understanding.

**Additional Help**
- Refer to the Qiskit Documentation: https://qiskit.org/documentation/
- For support, contact your course instructor or visit the IBM Quantum Community forums.

3. Or launch this lab directly now: [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jopaneur/QuantumAI-Labs/blob/main/Advanced_Labs/notebooks/Chapter_2_Operations_&_Scientific_Framework_of_QAIS_Advanced_Challenge_Bloch_Trajectories_Under_Composite_Gates.ipynb)



---
**Note for Lab Participants**
Each plot generated in this notebook is automatically saved as a `.png` file under: Advanced_Labs/figures/

The filenames follow the Appendix E figure numbering (e.g., `E2_1_Bloch_Trajectories.png`, `E2_6_DensityMatrix_Heatmap.png`).  

This allows you to both view results inline in Colab **and** find the corresponding image files for reports, submissions, or cross-references in the book.

**Where Figures Are Saved**
- In **Google Colab**, the images are created inside the session’s working directory at:  
  `/content/Advanced_Labs/figures/`  

- When running **locally**, they appear next to your notebook files, under the subfolder:  
  `Advanced_Labs/figures/`  

- These images are **not automatically added to your GitHub repo**. They will only appear there if you manually copy, commit, and push them.

**Customizing Save Location**
If you want the figures saved elsewhere, you can change the `subdir` default in the `save_e_figure()` helper or pass a different path each time you call it.


---

**Book Reference: Chapter 10 — Quantum Communication for Distributed AI Systems**

*Chapter 10* expands QAIS into the realm of quantum communication, introducing the principles that allow secure information exchange across distributed quantum networks. It explores how entanglement, measurement disturbance, and basis choice underpin quantum cryptographic protocols, where the act of eavesdropping leaves a measurable trace.
The BB84 Quantum Key Distribution (QKD) protocol exemplifies this principle: two parties—traditionally called Alice and Bob—use randomly chosen measurement bases to establish a shared cryptographic key while detecting any unauthorized interception attempts.

This chapter situates BB84 within the broader QAIS framework as an example of resilience through physics—security emerging not from computation, but from the laws of quantum mechanics themselves. It illustrates that trust in distributed AI systems must be enforced at the substrate level, where coherence, disturbance, and entanglement define both communication and protection.

**Beginner Lab 8 — BB84 Quantum Key Distribution: Error Rate Comparison**

Beginner Lab 8 brings these concepts to life by implementing the BB84 protocol in simulation. Learners perform key exchange with and without an eavesdropper (Eve) and compute the Quantum Bit Error Rate (QBER) to detect intrusion. The lab demonstrates that even simple quantum channels can self-verify integrity, forming the conceptual backbone for secure QAIS communication pipelines.

*Goal:* Simulate the BB84 quantum key distribution protocol to illustrate how eavesdropping affects key accuracy. Learners generate random qubit states in alternate bases, transmit them between Alice and Bob, and measure the resulting bit strings. The Quantum Bit Error Rate (QBER) is computed for both secure and intercepted channels.

**Expected Outcome**

* In the no-eavesdropper scenario, QBER remains low, confirming a stable, private key exchange.

* When Eve intercepts and resends, QBER rises sharply due to measurement disturbance, signaling detectable intrusion.

This lab reinforces Chapter 10’s principle that quantum communication is self-securing—its physics ensures that any attempt to observe or copy information changes the system in measurable ways. Cross-reference: Appendix E.1, Figure E.1.8 — QBER with and without Eavesdropping.

In [ ]:
# ---- Figure helper (robust; use in every coded lab) ----
import os, matplotlib.pyplot as plt

def save_e_figure(fig_label: str,
                  fname: str,
                  subdir: str = "Beginner_Labs/figures",
                  fig=None, ax=None):
    """Save the current/explicit figure with a prefixed label and consistent path."""
    os.makedirs(subdir, exist_ok=True)
    if fig is None:
        fig = plt.gcf()
    if ax is None:
        ax = fig.axes[0] if fig.axes else None
    if ax is None:
        print("⚠️ No axes found. Draw a plot first, or pass fig/ax explicitly.")
        return
    title = ax.get_title() or ""
    if not title.startswith(fig_label):
        ax.set_title((fig_label + " — " + title).strip(" —"))
    outpath = os.path.join(subdir, fname)
    fig.tight_layout()
    fig.savefig(outpath, dpi=160)
    print("Saved", outpath)


In [ ]:
# === Environment Setup ===
import sys, subprocess, pkgutil
def ensure(pkg):
    if pkg not in {m.name for m in pkgutil.iter_modules()}:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p in ['qiskit','qiskit-aer','matplotlib','numpy','scikit-learn']:
    ensure(p)
import qiskit, numpy as np, matplotlib.pyplot as plt
print('Python:', sys.version.split()[0])
print('Qiskit:', qiskit.__version__)


---

**Lab 8: BB84 Quantum Key Distribution — Error Rate Comparison**

**Detecting Eavesdroppers through Error Rates**

This lab simulates the BB84 quantum key distribution protocol. Alice sends qubits in random bases, Bob measures in random bases, and a shared key is established when their bases match. The experiment is run in two scenarios: without an eavesdropper and with an eavesdropper intercepting and resending the qubits. The bar chart compares the quantum bit error rate (QBER) between these scenarios. Without eavesdropping, the QBER is low, while with Eve present, the QBER rises significantly. This provides an intuitive proof that eavesdropping introduces detectable disturbance into the quantum channel. The visualization makes the abstract idea of “security by physics” accessible, as students can see how an adversary directly raises the error rates of shared keys.

*Book Reference: Chapter 10 — Quantum Communication for Distributed AI Systems*

QKD is demonstrated as a statistical protocol, comparing error rates in the presence or absence of eavesdropping. This reinforces Chapter 10’s role in linking secure communication to system-level resilience.

**Expected Results**

Without Eve: QBER is close to 0 (small statistical fluctuations only).

With Eve: QBER rises significantly, typically near 25% in ideal intercept–resend, since Eve’s random basis choices misalign with Alice’s half the time.

The bar chart should show a clear separation between the two cases, confirming QKD’s tamper-detection property.

In [ ]:
# Lab 8 — BB84 Quantum Key Distribution: Eavesdropping Detection

import numpy as np
import matplotlib.pyplot as plt

# --- Parameters ---
N = 2000
rng = np.random.default_rng(7)

# --- Alice's random bits and bases ---
bits_A  = rng.integers(0, 2, N)
bases_A = rng.integers(0, 2, N)

# --- Bob's random bases ---
bases_B = rng.integers(0, 2, N)

# --- Measurement function ---
def measure(bit: int, basis_alice: int, basis_bob: int) -> int:
    """Return Bob's measurement result. If bases match, outcome equals bit;
    if bases differ, outcome is random (0 or 1)."""
    return bit if basis_alice == basis_bob else rng.integers(0, 2)

# --- Case 1: Clean channel ---
bits_B = np.array([measure(b, ba, bb) for b, ba, bb in zip(bits_A, bases_A, bases_B)])
sift   = (bases_A == bases_B)  # sifted positions where bases matched
keyA   = bits_A[sift]
keyB   = bits_B[sift]
QBER   = np.mean(keyA != keyB)

# --- Case 2: Eve intercept-resend ---
bases_E = rng.integers(0, 2, N)
bits_E  = np.array([measure(b, ba, be) for b, ba, be in zip(bits_A, bases_A, bases_E)])
bits_B2 = np.array([measure(b, ba, bb) for b, ba, bb in zip(bits_E, bases_A, bases_B)])
keyB2   = bits_B2[sift]
QBER2   = np.mean(keyA != keyB2)

# --- Plot QBER comparison ---
plt.bar(["Without Eve", "With Eve"], [QBER, QBER2])
plt.ylabel("Quantum Bit Error Rate (QBER)")
plt.title("BB84: Eavesdropping Raises Error Rate")

# --- Save with correct IEEE label ---
fig, ax = plt.gcf(), plt.gca()
save_e_figure("Figure E.1.8", "P1_Lab08_BB84_QBER.png",
              subdir="Beginner_Labs/figures", fig=fig, ax=ax)

plt.show()


*Figure E.1.8 — BB84: Eavesdropping Raises Error Rate*
The bar chart compares the quantum bit error rate (QBER) in BB84 with and without an eavesdropper. Intercept–resend by Eve introduces basis mismatches that increase error rates, creating a clear, measurable signature of tampering.

**Methodology Analysis**

Alice encodes random bits in random bases (rectilinear or diagonal). Bob measures in random bases, producing results that match Alice’s bits only when their bases coincide. After basis reconciliation (sifting), they estimate the QBER on their shared key. To model eavesdropping, Eve intercepts Alice’s qubits, measures in random bases, and resends. This action introduces additional disturbances that inflate the QBER, making tampering detectable.

**Technical Analysis (for the Visual)**

In BB84, the sifted key should be error-free if no disturbance occurs. When Eve measures in a random basis, she collapses the quantum state into her chosen basis, disturbing half of the transmitted qubits. This error propagates into Alice and Bob’s sifted key, elevating the QBER. Mathematically, for intercept–resend, QBER ≈ 25%, which provides a reliable threshold to detect eavesdropping. The bar chart captures this security guarantee in a simple, empirical form.

**Intuition Sidebar**

Imagine Alice and Bob comparing notes with special-colored dice. When Eve secretly swaps in her own dice throws, mismatches appear more often. The errors are like footprints left behind — Alice and Bob can’t see Eve directly, but her tampering leaves a visible trail in the error rate.

**Conclusion — Lab 8**

This lab demonstrates BB84’s built-in eavesdropping detection. By simulating both clean and tampered channels, participants directly observe how intercept–resend raises the QBER, creating a tamper-evident key. This principle underlies the security of quantum key distribution and highlights why quantum measurements cannot be copied or intercepted without disturbance.

**Key Takeaways**

QBER is the diagnostic metric for detecting eavesdropping in QKD.

Without disturbance, sifted keys align almost perfectly; with Eve, error rates rise sharply.

Intercept–resend attacks introduce ~25% QBER, making tampering unmistakable.

BB84 leverages fundamental quantum principles to enforce communication integrity.

**Congratulations — Lab 8**

Congratulations on completing Lab 8 — BB84 Quantum Key Distribution! You’ve implemented and validated one of quantum cryptography’s most iconic protocols, gaining firsthand insight into how quantum uncertainty guarantees security. This knowledge equips you to appreciate and apply QKD concepts in secure communications and quantum AI network architectures.

### Appendix E → Appendix B Cross-Reference
See **Appendix B — Quick Self-Check, Chapter 10 — Quantum Communication for Distributed AI Systems**:  
- Questions 1–3 (QBER and eavesdropping detection).  
They review the security implications demonstrated in **E.2 Lab 3**.


---
**How to save or submit your work**

- **If you are a student (graded/evaluated):**  
  1. Export your key plots or the entire notebook to PDF (File → Print/Save as PDF).  
  2. Save the notebook (`.ipynb`).  
  3. Bundle any extra files (CSVs/images) if used.  
  4. Upload to your LMS or repository as instructed (include your name and lab number).  
  5. Repro checklist: set a random seed where applicable, note backend and shots, and list package versions.  

- **If you are a professional/self‑learner (non‑graded exercise):**  
  1. Save the notebook (`File → Download .ipynb`) to your computer for personal reference.  
  2. Optionally export to PDF for archiving.  
  3. Keep any generated plots or data locally.  
  4. Use version control (GitHub, GitLab) if you wish to track your personal progress.


---
